# Session 11 — NumPy & pandas for Research Data

> arrays, vectorizing, masks, axis, DataFrames, loc/iloc, groupby — then broadcasting, missing data, transform, melt, merge, time series.

Read each cell, **predict** the output, then run it with **Shift + Enter**. The practice (solutions collapsed) is at the end.

**Tips:** press **Tab** to autocomplete a name, **Shift + Tab** for a function's help. Need a library? Run `%pip install <name>` in a cell (e.g. `%pip install pandas`) — in the browser (JupyterLite) it fetches a Pyodide build and lasts for the session.

In [ ]:
# Setup (notebook only): write the data files this session reads.
from pathlib import Path
Path('students.csv').write_text('name,major,score\nAna,Education,91\nBen,Psychology,58\nCara,Education,73\nDev,Sociology,64\nEve,Psychology,88\nFinn,Education,79\n')
Path('survey.csv').write_text('respondent,q1_engagement,q2_clarity,q3_workload,q4_support\nR01,5,4,2,5\nR02,4,4,3,4\nR03,3,5,N/A,4\nR04,5,5,1,5\nR05,2,3,4,2\nR06,4,,3,4\nR07,5,4,2,5\nR08,1,2,5,1\n')
print('wrote students.csv, survey.csv')

In [ ]:
from io import StringIO

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

STUDENTS_CSV = """name,major,score
Ana,Education,91
Ben,Psychology,58
Cara,Education,73
Dev,Sociology,64
Eve,Psychology,88
Finn,Education,79
"""

SURVEY_CSV = """respondent,q1_engagement,q2_clarity,q3_workload,q4_support
R01,5,4,2,5
R02,4,4,3,4
R03,3,5,N/A,4
R04,5,5,1,5
R05,2,3,4,2
R06,4,,3,4
R07,5,4,2,5
R08,1,2,5,1
"""

In [ ]:
# --- 1. NumPy arrays: one type, one block of memory -----------------------
scores = np.array([91, 58, 73, 64, 88, 79])
print("array:", scores, "| dtype:", scores.dtype, "| shape:", scores.shape)
print("zeros:", np.zeros(3), "| arange:", np.arange(0, 10, 2), "| linspace:", np.linspace(0, 1, 5))

In [ ]:
# --- 2. Vectorized math: no loop needed ----------------------------------
print("curved +5:   ", scores + 5)                 # every element at once
print("as fraction: ", scores / 100)
print("list * 2:    ", [91, 58] * 2)               # a LIST repeats...
print("array * 2:   ", np.array([91, 58]) * 2)     # ...an ARRAY multiplies
print("mean / std:  ", scores.mean(), round(scores.std(), 2))

In [ ]:
# --- 3. Boolean masks: filter with a condition ---------------------------
passed = scores >= 60
print("mask:        ", passed)
print("passing:     ", scores[passed])
print("how many:    ", passed.sum())               # True counts as 1 (Session 2!)
print("60s and 70s: ", scores[(scores >= 60) & (scores < 80)])   # & not `and`, parentheses required
print("labels:      ", np.where(scores >= 60, "pass", "fail"))  # vectorized if/else

In [ ]:
# --- 4. 2D arrays and the axis argument ---------------------------------
# rows = 3 students, columns = 4 quizzes
quizzes = np.array([[8, 9, 7, 10],
                    [5, 6, 4, 7],
                    [9, 9, 10, 8]])
print("\nshape:", quizzes.shape)
print("one cell [1, 2]:", quizzes[1, 2])
print("student 0's row:", quizzes[0])
print("quiz 3 column:  ", quizzes[:, 3])
print("per quiz (axis=0):   ", quizzes.mean(axis=0))   # collapse rows -> one value per column
print("per student (axis=1):", quizzes.mean(axis=1))   # collapse columns -> one value per row

In [ ]:
# --- 5. pandas: load and inspect a DataFrame -----------------------------
students = pd.read_csv(StringIO(STUDENTS_CSV))
print()
print(students.head(3))
print("shape:", students.shape, "| columns:", list(students.columns))
print(students["score"].describe().round(1))       # count, mean, std, min, quartiles, max

In [ ]:
# --- 6. Selecting: columns, loc (labels), iloc (positions) ----------------
print(students["name"].tolist())                   # one column -> Series
print(students[["name", "score"]].head(2))         # list of columns -> DataFrame
print("loc[0, 'name']:", students.loc[0, "name"])
print(students.iloc[-1])                           # last row, by position

In [ ]:
# --- 7. Filtering rows -----------------------------------------------------
print(students[students["score"] >= 75])
print(students[(students["major"] == "Education") & (students["score"] < 80)])
print(students[students["major"].isin(["Sociology", "Psychology"])]["name"].tolist())
print(students.query("score > 70 and major != 'Education'"))

In [ ]:
# --- 8. New columns, sorting, counting ------------------------------------
students["passed"] = students["score"] >= 60
students["grade"] = pd.cut(students["score"], bins=[0, 59, 69, 79, 89, 100],
                           labels=["F", "D", "C", "B", "A"])
students["z"] = ((students["score"] - students["score"].mean()) / students["score"].std()).round(2)
print(students.sort_values("score", ascending=False))
print(students["major"].value_counts())

In [ ]:
# --- 9. groupby: Session 8's by-hand loop, in one line ---------------------
print(students.groupby("major")["score"].mean().round(1))
summary = students.groupby("major").agg(
    n=("name", "count"),
    mean_score=("score", "mean"),
    best=("score", "max"),
)
print(summary.round(1).sort_values("mean_score", ascending=False))

In [ ]:
# ============================== GOING DEEPER ==============================
print("\n--- GOING DEEPER ---")

In [ ]:
# --- deeper 1: views vs copies ----------------------------------------------
original = np.array([91, 58, 73, 64])
view = original[:2]          # a slice is a VIEW of the same memory
view[0] = 0
print("after editing the slice:", original)        # original changed!
safe = original[2:].copy()   # .copy() when you need independence
safe[0] = 99
print("after editing a copy:   ", original, "<- the 73 is untouched")

In [ ]:
# --- deeper 2: broadcasting -------------------------------------------------
# Shapes line up from the right; a size-1 (or missing) dimension stretches.
weights = np.array([0.1, 0.2, 0.3, 0.4])                # shape (4,) -> stretched over 3 rows
print("weighted totals:", (quizzes * weights).sum(axis=1).round(2))
z = (quizzes - quizzes.mean(axis=0)) / quizzes.std(axis=0)   # z-score every column at once
print("column z-scores:\n", z.round(2))

In [ ]:
# --- deeper 3: reproducible random numbers -----------------------------------
rng = np.random.default_rng(seed=42)               # same seed -> same numbers (methods sections!)
print("dice:", rng.integers(1, 7, size=8))
sim = rng.integers(1, 7, size=(100_000, 2)).sum(axis=1)
print("P(two dice sum to 7) ~", round((sim == 7).mean(), 3), "(exact: 0.167)")

In [ ]:
# --- deeper 4: missing data -------------------------------------------------
survey = pd.read_csv(StringIO(SURVEY_CSV), na_values=["N/A"])   # treat "N/A" as missing
print(survey.isna().sum())                         # missing per column
print("q2 dtype:", survey["q2_clarity"].dtype)     # float64: a NaN turns an int column into floats
items = [c for c in survey.columns if c.startswith("q")]
print(survey[items].mean().round(2))               # pandas skips NaN by default
print("rows complete:", len(survey.dropna()), "of", len(survey))
filled = survey.fillna(survey[items].median())     # or impute with each column's median
print(filled.loc[[2, 5]])

In [ ]:
# --- deeper 5: transform — group stats aligned to every row -----------------
students["major_mean"] = students.groupby("major")["score"].transform("mean").round(1)
students["vs_major"] = students["score"] - students["major_mean"]
print(students[["name", "major", "score", "major_mean", "vs_major"]])

In [ ]:
# --- deeper 6: reshape — melt (wide -> long) and pivot_table ----------------
long = survey.melt(id_vars="respondent", value_vars=items, var_name="item", value_name="rating")
print(long.head())
print(long.groupby("item")["rating"].agg(["mean", "count"]).round(2))
print(pd.crosstab(long["item"], long["rating"]))   # how many 1s, 2s, ... per item
print(students.pivot_table(index="major", columns="passed", values="score",
                           aggfunc="count", fill_value=0))

In [ ]:
# --- deeper 7: merge — join two tables on a key ------------------------------
advisors = pd.DataFrame({"major": ["Education", "Psychology", "History"],
                         "advisor": ["Dr. Lee", "Dr. Okafor", "Dr. Ruiz"]})
joined = students.merge(advisors, on="major", how="left", validate="many_to_one")
print(joined[["name", "major", "advisor"]])        # Sociology has no advisor -> NaN
check = students[["major"]].drop_duplicates().merge(advisors, on="major", how="outer", indicator=True)
print(check)                                       # _merge shows where each key came from

In [ ]:
# --- deeper 8: dates and time series ----------------------------------------
logins = pd.DataFrame({
    "day": pd.date_range("2026-09-01", periods=28, freq="D"),
    "logins": rng.integers(20, 60, size=28),
})
logins["weekday"] = logins["day"].dt.day_name()
weekly = logins.set_index("day")["logins"].resample("W").sum()
print(weekly)                                      # weeks end on Sunday; the last one is partial
logins["avg_7d"] = logins["logins"].rolling(7).mean().round(1)
print(logins.tail(3))

In [ ]:
# --- deeper 9: method chaining + the one assignment rule ---------------------
report = (
    students
    .query("passed")
    .groupby("major", as_index=False)
    .agg(n=("name", "count"), mean_score=("score", "mean"))
    .sort_values("mean_score", ascending=False)
    .round(1)
)
print(report)

# Change values through ONE .loc call — never df[mask]["col"] = ... (that edits a copy)
students.loc[students["score"] < 60, "grade"] = "D"   # a regrade: the F becomes a D
print(students[["name", "score", "grade"]])
print(students.to_csv(index=False)[:80], "...")     # to_csv("file.csv", index=False) writes a file

## Now you try

The **In class** tasks first, **In class — going deeper** in the second hour, and **Homework** before the next session.

## In class

Files provided: `students.csv`, `survey.csv` (the same data as Session 8).
Start every script with `import numpy as np` and `import pandas as pd`.

### Task 1 — Session 8, vectorized
Put the scores `[91, 58, 73, 64, 88, 79]` in a NumPy array. Without a loop, print: the
mean, every score curved by 5 points (capped at 100 with `np.minimum`), and how many
students passed (score >= 60).

### Task 2 — Quiz grid
Build a 3×4 array (3 students, 4 quizzes) of your own numbers. Print each student's mean
and each quiz's mean. Which `axis` gave which? Then print the single highest score and
**which student** got it (`np.argmax` on the right axis).

### Task 3 — Load and look
`pd.read_csv("students.csv")`. Print `.shape`, `.head()`, `.info()`, and
`["score"].describe()`. Compare the mean to your Session 8 hand-computed one.

### Task 4 — Filter and label
Show only Education students scoring under 80. Then add a `grade` column with `pd.cut`
(A ≥ 90, B ≥ 80, C ≥ 70, D ≥ 60, else F) and print the students sorted by score.

### Task 5 — Descriptives by major
With one `groupby(...).agg(...)`, build a table with `n`, `mean_score`, `min_score`,
`max_score` per major, sorted by `mean_score`.

### Trap check
Predict, then run: `np.array([60, 75]) > 70 and np.array([60, 75]) < 90`. What
happens, and what should you write instead?

### Bonus — Pythonic idiom drill
Cover the `# ->` answers, predict each line, then run.

```python
import numpy as np
a = np.array([3, 1, 2])
print(a * 2)                  # -> [6 2 4]        (a list would repeat: [3, 1, 2, 3, 1, 2])
print(a[a > 1])               # -> [3 2]          (mask: keep where True)
print(np.sort(a)[::-1])       # -> [3 2 1]        (sort, then reverse)
print((a > 1).mean())         # -> 0.666...       (share of True values)
```

## In class — going deeper (second hour)

### Task 1 — z-scores by broadcasting
For your quiz grid from Task 2, compute every cell's z-score **within its quiz column**
in one expression. Check that each column's mean is now ~0.

### Task 2 — A view surprise
Make `a = np.arange(5)`, take `b = a[1:3]`, set `b[:] = 0`. Predict `a`. Redo it with
`.copy()` and confirm `a` survives.

### Task 3 — Clean the survey
Read `survey.csv` so that `"N/A"` becomes missing. Print missing values per column, the
mean of each `q*` item, and the number of complete rows. Why did `q2_clarity` become
`float64`?

### Task 4 — Melt it long
Melt the survey to one row per (respondent, item, rating). Produce a table of `mean` and
`count` per item, and a `crosstab` of item × rating.

### Task 5 — Who's above their major?
Add `major_mean` with `groupby(...).transform("mean")`, then list the students who beat
their own major's average.

### Task 6 — Join in advisors
Create `advisors` (Education → Dr. Lee, Psychology → Dr. Okafor). Left-merge it onto the
students with `validate="many_to_one"`. Which students got `NaN`? Re-run with
`indicator=True` to confirm.

### Task 7 — Weekly logins
With `rng = np.random.default_rng(1)`, build 21 days of logins (`pd.date_range` +
`rng.integers(10, 50, size=21)`). Print weekly totals with `resample("W")` and a 3-day
rolling mean.

## Homework (before the capstone)

*~30–45 minutes, outside class — it doesn't count toward class time. Try everything before peeking at the solutions.*

### Task 1 — Rewrite Session 8 in pandas
Take your Session 8 survey summary (per-item mean and `n_valid`, written to
`survey_summary.csv`). Redo it with pandas in **at most six lines**. Diff the two output
files: do the numbers match?

### Task 2 — A simulation
With a seeded generator, simulate 10,000 classes of 30 students whose scores are
`rng.normal(72, 12)` (clip to 0–100). What fraction of classes have a mean **above 75**?
Run it twice with the same seed, then with a different one.

### Task 3 — Your own data
Load a CSV from your own work with `pd.read_csv`. Run `.info()` and `.isna().sum()`,
then produce one `groupby` table you'd put in a paper. Write down every cleaning decision
you made (dropped? imputed? renamed?), because that list *is* part of your methods section.

In [ ]:
# Your practice work — type here. Predict before you run.


<details>
<summary><strong>Show solutions</strong></summary>

### In class

```python
import numpy as np
import pandas as pd

# Task 1
scores = np.array([91, 58, 73, 64, 88, 79])
print(scores.mean())                      # 75.5
print(np.minimum(scores + 5, 100))        # [96 63 78 69 93 84]
print((scores >= 60).sum())               # 5

# Task 2
q = np.array([[8, 9, 7, 10], [5, 6, 4, 7], [9, 9, 10, 8]])
print(q.mean(axis=1))                     # per student (columns collapse)
print(q.mean(axis=0))                     # per quiz    (rows collapse)
print(q.max(), "by student", q.max(axis=1).argmax())   # 10 by student 0 (first max wins)

# Task 3
students = pd.read_csv("students.csv")
print(students.shape)                     # (6, 3)
students.info()
print(students["score"].describe())       # mean 75.5 — same as Session 8

# Task 4
print(students[(students["major"] == "Education") & (students["score"] < 80)])
students["grade"] = pd.cut(students["score"], bins=[0, 59, 69, 79, 89, 100],
                           labels=["F", "D", "C", "B", "A"])
print(students.sort_values("score", ascending=False))

# Task 5
table = students.groupby("major").agg(
    n=("name", "count"), mean_score=("score", "mean"),
    min_score=("score", "min"), max_score=("score", "max"),
).sort_values("mean_score", ascending=False)
print(table)
```

Trap: `ValueError: The truth value of an array with more than one element is ambiguous`.
`and` needs ONE True/False, but each side is a whole array. Write
`(a > 70) & (a < 90)`, with the parentheses, because `&` binds tighter than `>`.

### In class — going deeper

```python
# Task 1
z = (q - q.mean(axis=0)) / q.std(axis=0)  # (3,4) - (4,) broadcasts down the rows
print(z.mean(axis=0).round(10))           # all zero (a "-0." is still zero)

# Task 2
a = np.arange(5); b = a[1:3]; b[:] = 0
print(a)                                  # [0 0 0 3 4] — b was a view
a = np.arange(5); b = a[1:3].copy(); b[:] = 0
print(a)                                  # [0 1 2 3 4]

# Task 3
survey = pd.read_csv("survey.csv", na_values=["N/A"])
items = [c for c in survey.columns if c.startswith("q")]
print(survey.isna().sum())
print(survey[items].mean().round(2))
print(len(survey.dropna()))               # 6
# q2 has a blank -> NaN, and NaN is a float, so the whole column becomes float64.

# Task 4
long = survey.melt(id_vars="respondent", value_vars=items, var_name="item", value_name="rating")
print(long.groupby("item")["rating"].agg(["mean", "count"]).round(2))
print(pd.crosstab(long["item"], long["rating"]))

# Task 5
students["major_mean"] = students.groupby("major")["score"].transform("mean")
print(students[students["score"] > students["major_mean"]]["name"].tolist())   # ['Ana', 'Eve']

# Task 6
advisors = pd.DataFrame({"major": ["Education", "Psychology"], "advisor": ["Dr. Lee", "Dr. Okafor"]})
joined = students.merge(advisors, on="major", how="left", validate="many_to_one", indicator=True)
print(joined[["name", "major", "advisor", "_merge"]])   # Dev (Sociology) -> NaN, left_only

# Task 7
rng = np.random.default_rng(1)
logins = pd.Series(rng.integers(10, 50, size=21),
                   index=pd.date_range("2026-09-01", periods=21, freq="D"))
print(logins.resample("W").sum())
print(logins.rolling(3).mean().round(1).tail())
```

### Homework

```python
# Task 1 — six lines
survey = pd.read_csv("survey.csv", na_values=["N/A"])
items = [c for c in survey.columns if c.startswith("q")]
out = pd.DataFrame({"item": items,
                    "mean": survey[items].mean().round(2).values,
                    "n_valid": survey[items].count().values})
out.to_csv("survey_summary_pandas.csv", index=False)
print(out)

# Task 2
rng = np.random.default_rng(7)
classes = np.clip(rng.normal(72, 12, size=(10_000, 30)), 0, 100)   # rows = classes
print((classes.mean(axis=1) > 75).mean())   # ~0.08: a class mean above 75 is rare
```

Task 3 is yours: the pattern is `read_csv` → `info()` / `isna().sum()` → clean →
`groupby(...).agg(...)`, plus the written list of decisions.

</details>